# CV Lightcurve Downloader & Plotter

## Step 1: Download and Plot TESS Lightcurves

This notebook downloads TESS lightcurve data for cataclysmic variables (CVs) and creates basic plots for visual inspection.

### Features:
- **TESS Data Download**: Automatic retrieval of lightcurve data
- **Data Cleaning**: Basic outlier removal and NaN handling  
- **Visualization**: Raw lightcurve plotting with sector information
- **Multi-sector Support**: Combines data from multiple TESS sectors

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
import warnings
warnings.filterwarnings('ignore')
from lightkurve import LightCurve
from astropy.timeseries import LombScargle

# Enable interactive plotting
%matplotlib widget

# Target Configuration - Change these for your target
TIC_ID = 372519345
TARGET_NAME = "OY Carinae"
NUM_OF_SECTORS = 5  # Number of sectors to download, set to None for all

print(f"🎯 Target: {TARGET_NAME} (TIC {TIC_ID})")
print("📡 Ready to download TESS lightcurve data...")

## Download TESS Data

Search for and download all available TESS lightcurve data for the target.

In [ ]:
# Search for TESS data
search_result = lk.search_lightcurve(f'TIC {TIC_ID}', mission='TESS', author='SPOC', cadence='short')

if len(search_result) == 0:
    raise ValueError(f"❌ No TESS data found for TIC {TIC_ID}")

print(f"✅ Found {len(search_result)} lightcurve file(s)")

# Download all available lightcurves
print("\n💾 Downloading lightcurve data...")
if type(NUM_OF_SECTORS) == int:
    lc_collection = search_result[:NUM_OF_SECTORS].download_all()
    lc : LightCurve = lc_collection.stitch()
else:
    lc_collection = search_result.download_all()
    lc : LightCurve = lc_collection.stitch()

lc = lc.remove_nans()
  # Remove NaNs from the light curve
print(f"✅ Downloaded {len(lc_collection)} sector(s)")
print(f"📊 Total number of data points: {len(lc)}")
print(f"⏳ Amount of time: {np.median(np.diff(lc.time.value)) * len(lc)} days")

flattened_lc, trend = lc.flatten(window_length=1001, return_trend=True, break_tolerance=1)

# Create a quick preview plot of the downloaded data
fig, ax = plt.subplots(figsize=(10, 4))

# Plot the raw lightcurve
ax.scatter(lc.time.value, lc.flux.value, s=0.5, alpha=0.7, color='royalblue', rasterized=True)
# Plot the flattened lightcurve
ax.scatter(flattened_lc.time.value, flattened_lc.flux.value, color='black', s = 0.5, alpha = 0.7, label='Flattened Light Curve', rasterized=True)
# Plot the trend
# ax.scatter(trend.time.value, trend.flux.value, color='red', s=0.5, alpha=0.7, label='Trend', rasterized=True)

# Formatting
ax.set_xlabel('Time (BTJD)', fontsize=11)
ax.set_ylabel('Normalized Flux', fontsize=11)
ax.set_title(f'Quick Preview: {TARGET_NAME} (TIC {TIC_ID})', fontsize=12)
ax.grid(True, alpha=0.3)

# Add sector information
ax.text(0.02, 0.95, f'Sectors: {len(lc_collection)}\nPoints: {len(lc):,}', 
    transform=ax.transAxes, fontsize=10,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

In [ ]:
from scipy.interpolate import CubicSpline

dt = 10.000000 * 1/(24*3600)
t = lc.time.value
flux = lc.flux.value

spline = CubicSpline(t, flux)

In [ ]:
gap_threshold = np.median(np.diff(t)) * 5
breaks = np.where(np.diff(t) > gap_threshold)[0] + 1
segments = np.split(np.arange(t.size), breaks)

t_even_parts, y_even_parts = [], []
for segment in segments:
    ti, yi = t[segment], flux[segment]

    te = np.arange(ti.min(), ti.max(), dt)
    spline = CubicSpline(ti, yi)
    ye = spline(te)

    t_even_parts.append(te)
    y_even_parts.append(ye)

t_even = np.concatenate(t_even_parts)
y_even = np.concatenate(y_even_parts)

new_lc = LightCurve(time=t_even, flux=y_even)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
# Plot the evenly sampled lightcurve
ax.scatter(new_lc.time.value, new_lc.flux.value, s=0.5, alpha=0.7, color='orange', rasterized=True)
# Formatting
ax.set_xlabel('Time (BTJD)', fontsize=11)
ax.set_ylabel('Normalized Flux', fontsize=11)
ax.set_title(f'Evenly Sampled Lightcurve: {TARGET_NAME} (TIC {TIC_ID})', fontsize=12)
ax.grid(True, alpha=0.3)
# Add sector information
plt.show()

In [ ]:
np.median(np.diff(new_lc.time.value)) * 24 * 3600  # Convert to seconds

In [ ]:
for i in range(10, 2000, 10000):
    print(np.diff(new_lc.time.value)[i] * 24 * 3600)  # Convert to seconds

In [ ]:
lc.interact_bls()

In [ ]:
lc.to_periodogram().plot()
new_lc.to_periodogram().plot()

In [ ]:
np.arange()

In [ ]:
breaks

In [ ]:
t[breaks]

In [ ]:
np.split(t, )

In [ ]:
new_lc.plot()

In [ ]:
lc.to_periodogram(method='lombscargle').plot()

In [ ]:
periodogram = lc.to_periodogram(method='lombscargle')/10000000

In [ ]:
# ------------- compute window as before -------------
wfreq, wpow = LombScargle(lc.time.value,
                          np.ones_like(lc.time.value)).autopower()

# ------------- normalise & plot on log scale ----------
plt.figure(figsize=(10, 4))
wpow_norm = wpow / wpow.max()             # brings peak to 1
plt.semilogy(periodogram.frequency, periodogram.power, 'k', lw=1)  # black – real
plt.semilogy(wfreq, wpow_norm, 'r', lw=1)
plt.xlim(10, 20)                            # show 1‑d⁻¹ region first
plt.ylim(1e-10, 1e-6)
plt.xlabel("Frequency [d$^{-1}$]")
plt.ylabel("Spectral‑window power (normalised)")

In [ ]:
# index of tallest black peak
i_true = np.argmax(periodogram.power)
f_true = periodogram.frequency[i_true]

# find alias family: f_true ± n*f_alias  (e.g. f_alias = 1 d^-1)
f_alias = 1.0            # or use 0.5 if you see half-day aliases
for n in range(-5, 6):
    f_test = f_true + n*f_alias
    j = np.argmin(np.abs(periodogram.frequency - f_test))
    print(f"{n:+d} alias at {periodogram.frequency[j]:.5f} d^-1 with power {periodogram.power[j]:.2e}")

In [ ]:
P_orb   = 1/15.50317  # days
t0      = 1571.14083  # reference mid‑eclipse (HJD or BTJD)
half_width = 0.05      # in phase (±5 % of an orbit)

phase = ((lc.time.value - t0) % P_orb) / P_orb          # [0,1)
phase[phase > 0.5] -= 1             

In [ ]:
len(phase)

In [ ]:
len(phase)

plt.figure(figsize=(10, 4))
plt.plot(lc.time.value, phase)
plt.scatter(lc.time.value, phase, s=0.5, alpha=0.7, color='royalblue', rasterized=True)

In [ ]:
# Stitch sectors together and clean data
print("🔗 Stitching sectors together...")
lc = lc_collection.stitch()

print("🧹 Cleaning data (removing NaNs and outliers)...")
lc_clean = lc.remove_nans().remove_outliers(sigma=5)

# Print summary statistics
print(f"\n📊 Data Summary:")
print(f"   • Total data points: {len(lc_clean):,}")
print(f"   • Time span: {lc_clean.time.max() - lc_clean.time.min():.1f} days")
print(f"   • Cadence: {(lc_clean.time[1] - lc_clean.time[0])*24*60:.1f} minutes")
print(f"   • Sectors: {len(lc_collection)}")

# Store the clean lightcurve for plotting
lightcurve = lc_clean

In [ ]:
## Plot Raw Lightcurve

Visualize the downloaded and cleaned lightcurve data.

In [ ]:
# Plot the full lightcurve
fig, ax = plt.subplots(figsize=(14, 6))

# Plot data points
ax.scatter(lightcurve.time.value, lightcurve.flux.value, 
          s=0.5, alpha=0.7, color='steelblue', rasterized=True)

# Formatting
ax.set_xlabel('Time (BTJD)', fontsize=12)
ax.set_ylabel('Normalized Flux', fontsize=12)
ax.set_title(f'{TARGET_NAME} (TIC {TIC_ID}) - TESS Lightcurve', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add text with data info
ax.text(0.02, 0.98, f'Data points: {len(lightcurve):,}\nTime span: {lightcurve.time.max() - lightcurve.time.min():.1f} days', 
        transform=ax.transAxes, verticalalignment='top', 
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Plot individual sectors
n_sectors = len(lc_collection)
fig, axes = plt.subplots(n_sectors, 1, figsize=(14, 3*n_sectors), sharex=False)

# Handle case of single sector
if n_sectors == 1:
    axes = [axes]

for i, lc_sector in enumerate(lc_collection):
    # Clean individual sector
    lc_sector_clean = lc_sector.remove_nans().remove_outliers(sigma=5)
    
    # Plot
    axes[i].scatter(lc_sector_clean.time.value, lc_sector_clean.flux.value, 
                   s=0.8, alpha=0.7, color='steelblue', rasterized=True)
    
    # Formatting
    axes[i].set_ylabel('Normalized Flux', fontsize=10)
    axes[i].set_title(f'Sector {lc_sector.meta.get("SECTOR", i+1)} - {len(lc_sector_clean):,} points', 
                     fontsize=11)
    axes[i].grid(True, alpha=0.3)

# Set x-label only for bottom plot
axes[-1].set_xlabel('Time (BTJD)', fontsize=12)

plt.suptitle(f'{TARGET_NAME} - Individual TESS Sectors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Display detailed statistics
print("📊 DETAILED DATA STATISTICS")
print("=" * 50)

print(f"🎯 Target: {TARGET_NAME} (TIC {TIC_ID})")
print(f"📡 Mission: TESS")
print(f"📁 Total Sectors: {len(lc_collection)}")

print(f"\n⏱️ TIMING INFORMATION:")
print(f"   • Time range: {lightcurve.time.min().value:.2f} - {lightcurve.time.max().value:.2f} BTJD")
print(f"   • Time span: {(lightcurve.time.max() - lightcurve.time.min()).value:.2f} days")
print(f"   • Median cadence: {np.median(np.diff(lightcurve.time.value))*24*60:.1f} minutes")

print(f"\n📈 FLUX STATISTICS:")
print(f"   • Data points: {len(lightcurve):,}")
print(f"   • Mean flux: {np.mean(lightcurve.flux.value):.6f}")
print(f"   • Flux std: {np.std(lightcurve.flux.value):.6f}")
print(f"   • Flux range: {np.min(lightcurve.flux.value):.6f} - {np.max(lightcurve.flux.value):.6f}")
print(f"   • RMS scatter: {np.std(lightcurve.flux.value)/np.mean(lightcurve.flux.value)*1e6:.1f} ppm")

print(f"\n💾 DOWNLOAD COMPLETE!")
print("🎉 Ready for further analysis...")

## Next Steps

The lightcurve data has been successfully downloaded and visualized! 

### What we've accomplished:
- ✅ Downloaded TESS data for the target
- ✅ Cleaned and processed the lightcurve  
- ✅ Created overview and sector-by-sector plots
- ✅ Generated detailed statistics

### Future analysis steps could include:
- 🔍 Period analysis and periodogram generation
- 🌟 Eclipse detection and timing
- 📊 O-C diagram creation  
- 📈 Long-term period evolution studies

The data is now ready for further CV analysis!